# Limpieza de los Datos de la Tabla bronce.prestamos para Cargalos en la Capa Plata

Proposito del script:  
- Verificar columna por columna los tipos de datos para encontrar inconsistencias en los datos.  
- Limpiar y estandarizar, columna por columna los datos.
- Crear un archivo para los registros con inconsistencias con nombre "revision_prestamos.parquet".  
- Exportar la nueva tabla como un archivo con nombre "semi_limpio_prestamos.parquet".

# Estableciendo la Conexion

In [1]:
# Importando las librerias y creando la conexion 
import pandas as pd 
import numpy as np 
from datetime import date
from funciones import limpiar_texto,formato_clasificacion_riesgo_sbs,asignacion_estado_prestamo
from conexiones_y_rutas import obtener_engine,obtener_ruta_archivo
engine = obtener_engine()

df_prestamos = pd.read_sql(
    "SELECT * FROM bronce.prestamos",
    con= engine
)

df_prestamos_tra = df_prestamos.copy()

# Archivos de Ayuda

In [2]:
df_productos_crediticios= pd.read_parquet(
    obtener_ruta_archivo("archivos_limpios","limpio_productos_crediticios.parquet")
)
df_productos_crediticios_tra = df_productos_crediticios.copy()
df_productos_crediticios_tra.head()

,producto_id,nombre_producto,tipo_credito,tasa_nom_min,tasa_nom_max,plazo_min_meses,plazo_max_meses,monto_minimo,monto_maximo,requiere_garantia,moneda
0,1,Crédito Personal Libre Disponibilidad,Personal,18.0,36.0,6,60,1000.0,50000.0,False,PEN
1,2,Crédito Hipotecario Vivienda,Hipotecario,7.5,11.0,60,360,50000.0,1000000.0,True,PEN
2,3,Crédito Vehicular,Vehicular,2.5,18.0,12,72,10000.0,200000.0,True,PEN
3,4,Crédito MYPE Capital De Trabajo,Microempresa,20.0,48.0,6,48,2000.0,100000.0,False,PEN
4,5,Crédito de Consumo,Consumo,24.0,42.0,3,36,500.0,20000.0,False,PEN


In [3]:
df_sucursales = pd.read_parquet(obtener_ruta_archivo("archivos_limpios","limpio_sucursales.parquet"),
    columns=["sucursal_id","fecha_apertura"])
df_sucursales_tra = df_sucursales.copy()
df_sucursales_tra.head()

,sucursal_id,fecha_apertura
0,1,2005-07-24
1,2,2010-01-03
2,3,2007-04-20
3,4,2010-02-19
4,5,2007-02-07


In [4]:
df_clientes = pd.read_parquet(obtener_ruta_archivo("archivos_semi_limpios","semi_limpio_clientes.parquet"))
df_clientes_tra = df_clientes.copy()
df_clientes_tra.head()

,cliente_id,tipo_documento,numero_documento,nombres,apellido_paterno,apellido_materno,fecha_nacimiento,edad,genero,estado_civil,...,ingresos_mensuales,egresos_mensuales,patrimonio_estimado,score_crediticio,segmento_cliente,canal_captacion,antiguedad_cliente_meses,fecha_registro,sucursal_id,estado_cliente
0,1,DNI,60366909,Carmen,Ramírez,Flores,1977-09-16,48,Femenino,Casado,...,2072.19,888.76,39631.77,550,Regular,Agencia,122,2016-05-07,14,Activo
1,2,DNI,62729806,Cecilia,Cusi,Cusi,1964-03-13,62,Femenino,Casado,...,5663.68,3037.05,125691.99,528,Regular,Digital,105,2017-10-18,13,Activo
2,3,CE,641708053,Juan,Ortiz,Medina,1964-08-01,62,Masculino,Casado,...,4750.66,2302.13,226055.14,493,Regular,Agencia,90,2019-01-08,13,Activo
3,4,DNI,29912419,Carlos,Flores,Morales,1976-11-09,49,Masculino,Soltero,...,1832.75,763.53,104986.09,490,Regular,Telemarketing,196,2010-03-12,15,Activo
4,5,DNI,86518506,Sandra,González,Morales,1980-11-20,45,Femenino,Viudo,...,850.00,405.03,37044.25,392,Regular,Digital,162,2013-01-08,4,Activo


In [5]:
df_oficial = pd.read_parquet(obtener_ruta_archivo("archivos_semi_limpios","semi_limpio_oficiales_credito.parquet"),
    columns=["oficial_id","fecha_ingreso"])
df_oficial_tra = df_oficial.copy()
df_oficial_tra.head()

,oficial_id,fecha_ingreso
0,1,2017-12-20
1,2,2015-06-29
2,3,2010-09-20
3,4,2021-03-13
4,5,2022-08-04


# Resumen de las Columnas

- **prestamo_id**: Identificador unico de cada prestamo.  
- **cliente_id**: Identificador del cliente asociado al prestamo.  
- **sucursal_id**: Identificador de la sucursal asociada al prestamo.  
- **producto_id**: Identificador del producto asociado al prestamo.  
- **oficial_id**:  Identificador del oficial de credito asociado al prestamo.  
- **numero_contrato**: Numero de contrato del prestamo "CONT-000000(ID_PRESTAMO)" => Maximo 13 caracteres 
- **fecha_otorgamiento**: Fecha de otorgamiento del credito.   
- **fecha_vencimiento**: Fecha de vencimiento del credito.  
- **monto_original**: Monto Original del credito.  
- **saldo_capital_vigente**: Saldo Capital Vigente.   
- **tasa_interes_nominal_anual**:  Tasa nominal anual del credito.  
- **tasa_interes_efectiva_anual**:  Tasa efectiva anual del credito.  
- **plazo_meses**: Plazo en meses para cancelar el credito.  
- **tipo_credito**: Tipo de Credito (ejem: Personal, Microempresa o Hipotecario).  
- **moneda**: Moneda del credito (ejem: USD o PEN).  
- **frecuencia_pago**: Frecuencia on que se realizan los pagos (ejem: Mensual).  
- **cuota_programada**: Monto a pagar en cada cuota. 
- **numero_cuotas_total**: Numero de cuotas totales del credito.  
- **numero_cuotas_pagadas**: Numero de cuotas pagadas.  
- **numero_cuotas_pendientes**: Numero de cuotas pendientes.  
- **estado**: Estado del credito (ejem: Vigente o Moroso).    
- **dias_mora**: Dias de mora.  
- **clasificacion_riesgo_sbs**: Casificacion de riesgo de la sbs (ejem: Normal, CPP o Perdida).  
- **garantia_tipo**: Tipo de garantia entregada (ejem: Aval, Carta Fianza o Sin Garantia ).  
- **garantia_valor**: Valor de la garantia.  
- **proposito_credito**: Proposito del credito (ejem: Viaje / Turismo, Mejoras Del Hogar o Expansión De Negocio).    
- **canal_desembolso**: Canal por el cual realiza el desembolso (ejem: Transferencia Bancaria, Agencia o Cheque De Gerencia).  
- **fecha_primer_pago_programado**: Fecha del primer pago programado. 
- **fecha_ultimo_pago_real**: Fecha donde se realizo el ultimo pago del crédito.  

# Verificacion de la Calidad y Limpieza de los Datos

In [6]:
df_prestamos_tra.info()  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6499 entries, 0 to 6498
Data columns (total 29 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   prestamo_id                   6499 non-null   int64  
 1   cliente_id                    6499 non-null   int64  
 2   sucursal_id                   6499 non-null   int64  
 3   producto_id                   6499 non-null   int64  
 4   oficial_id                    6499 non-null   int64  
 5   numero_contrato               6499 non-null   object 
 6   fecha_otorgamiento            6499 non-null   object 
 7   fecha_vencimiento             6499 non-null   object 
 8   monto_original                6499 non-null   float64
 9   saldo_capital_vigente         6499 non-null   float64
 10  tasa_interes_nominal_anual    6499 non-null   float64
 11  tasa_interes_efectiva_anual   6499 non-null   float64
 12  plazo_meses                   6499 non-null   int64  
 13  tip

In [7]:
df_prestamos_tra.head()

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real
0,1,4848,23,6,60,CONT-00000001,2023-05-07,2025-10-23,9218.73,3596.11,...,10,Vigente,0,Normal,None,0.0,Viaje / Turismo,Transferencia Bancaria,2023-06-06,27-12-2024
1,2,44,5,1,21,CONT-00000002,2021/02/12,2025-07-21,7767.38,1473.65,...,7,Vigente,0,Normal,Aval,0.0,Mejoras del Hogar,Agencia,2021-03-14,2024-12-23
2,3,2474,16,1,35,CONT-00000003,2023-11-22,2027-11-01,12512.04,10640.44,...,35,Vigente,0,Normal,Aval,0.0,Mejoras del Hogar,Transferencia Bancaria,22-12-2023,16/12/2024
3,4,637,15,4,32,CONT-00000004,19/04/2024,2024-10-16,3555.11,0.00,...,0,Cancelado,0,Normal,Carta Fianza,0.0,Expansión de Negocio,Agencia,19/05/2024,2024/10/16
4,5,3622,10,4,12,CONT-00000005,19-04-2020,2022-10-06,8388.15,0.00,...,0,Cancelado,0,Normal,Sin Garantía,0.0,Capital de Trabajo,Agencia,19/05/2020,2022-10-06


In [8]:
# Registros Duplicados 
df_prestamos_tra[df_prestamos_tra.duplicated(keep=False)]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


In [9]:
# Elimina duplicados 
df_prestamos_tra.drop_duplicates(inplace=True)
df_prestamos_tra.reset_index(inplace=True,drop=True)

## prestamo_id

In [10]:
# Verifica que no existan ids duplicados de prestamo 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra[df_prestamos_tra.prestamo_id.duplicated(keep=False)]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real
201,202,2531,17,5,52,CONT-00000202,28/08/2023,19-05-2025,6363.54,1954.81,...,5,Vigente,0,Normal,Aval,0.00,Electrodomésticos,Transferencia Bancaria,27/09/2023,20/12/2024
871,872,23,6,5,40,CONT-00000872,2021/11/20,2024-02-08,4271.14,1234.00,...,0,Moroso,354,CPP,Aval,0.00,Muebles y Enseres,Cheque de Gerencia,2021-12-20,2024-02-08
1679,1680,1419,6,3,41,CONT-00001680,2020/11/10,13/04/2026,33391.44,11075.01,...,16,Vigente,0,Normal,None,62577.71,Maquinaria,Agencia,2020-12-10,19-12-2024
4164,4165,2821,23,1,60,CONT-00004165,2021/02/13,27-07-2024,6207.18,0.00,...,0,Cancelado,0,Normal,Sin Garantía,0.00,None,Transferencia Bancaria,15/03/2021,2024-07-27
6495,4165,2821,23,1,60,CONT-00004165,2021/02/13,27-07-2024,6207.18,0.00,...,0,Cancelado,0,Normal,Sin Garantía,0.00,None,Transferencia Bancaria,2021-03-15,2024-07-27
6496,872,23,6,5,40,CONT-00000872,20/11/2021,2024/02/08,4271.14,1234.00,...,0,Moroso,354,CPP,Aval,0.00,Muebles y Enseres,Cheque de Gerencia,20/12/2021,2024-02-08
6497,202,2531,17,5,52,CONT-00000202,28/08/2023,2025-05-19,6363.54,1954.81,...,5,Vigente,0,Normal,Aval,0.00,Electrodomésticos,Transferencia Bancaria,2023/09/27,2024-12-20
6498,1680,1419,6,3,41,CONT-00001680,2020-11-10,13/04/2026,33391.44,11075.01,...,16,Vigente,0,Normal,None,62577.71,Maquinaria,Agencia,2020/12/10,19/12/2024


In [11]:
# Verifica si existen ids negativos o 0 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra[df_prestamos_tra.prestamo_id <= 0]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


## cliente_id

In [12]:
# Verifica que cliente_id sea valido 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra[df_prestamos_tra.cliente_id <= 0]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


In [13]:
# Verifica que cliente_id exista en la tabla de clientes 
# Resultados Esperados: both: 6499, left_only: 0, right_only: 0
verificar_cliente_id = df_prestamos_tra.merge(
    right=df_clientes_tra,
    on='cliente_id',
    how='left',
    indicator=True
)

verificar_cliente_id._merge.value_counts()

_merge
both          6499
left_only        0
right_only       0
Name: count, dtype: int64

## sucursal_id

In [14]:
# Verifica que sucursal_id sea valido 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra[df_prestamos_tra.sucursal_id <= 0]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


In [15]:
# Verifica que sucursal_id exista en la tabla de sucursales 
# Resultados Esperados: both: 6499, left_only: 0, right_only: 0
verificar_sucursal_id = df_prestamos_tra.merge(
    right=df_sucursales_tra,
    on='sucursal_id',
    how='left',
    indicator=True
)

verificar_sucursal_id._merge.value_counts()

_merge
both          6499
left_only        0
right_only       0
Name: count, dtype: int64

## producto_id

In [16]:
# Verifica que producto_id sea valido 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra[df_prestamos_tra.producto_id <= 0]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


In [17]:
# Verifica que producto_id exista en la tabla de productos 
# Resultados Esperados: both: 6499, left_only: 0, right_only: 0
verificar_producto_id = df_prestamos_tra.merge(
    right=df_productos_crediticios_tra,
    on='producto_id',
    how='left',
    indicator=True
)

verificar_producto_id._merge.value_counts()

_merge
both          6499
left_only        0
right_only       0
Name: count, dtype: int64

## oficial_id

In [18]:
# Verifica que oficial_id sea valido 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra[df_prestamos_tra.oficial_id <= 0]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


In [19]:
# Verifica que oficial_id exista en la tabla de oficiales_credito 
# Resultados Esperados: both: 6499, left_only: 0, right_only: 0
verificar_oficial_id = df_prestamos_tra.merge(
    right= df_oficial_tra,
    on='oficial_id',
    how='left',
    indicator=True
)

verificar_oficial_id._merge.value_counts()

_merge
both          6499
left_only        0
right_only       0
Name: count, dtype: int64

## numero_contrato  

In [20]:
# Verfica que tengan el formato correcto y la extencion correcta
# Resultados Esperados: Tabla Vacia
df_prestamos_tra.numero_contrato[
    (df_prestamos_tra.numero_contrato
        != df_prestamos_tra.numero_contrato.str.strip().str.upper())
    |
    (df_prestamos_tra.numero_contrato.apply(len)!= 13)
    |
    (df_prestamos_tra.numero_contrato.isna())
]

Series([], Name: numero_contrato, dtype: object)

In [21]:
# Verifica el formato de la referencia del contrato "CONT-000000(ID_PRESTAMO)" => Maximo 13 caracteres 

# 'CONT-'+ RELLENAR_CON_CEROS + ID_PRESTAMO => Maximo 13 caracteres
num_pag_gene = (
    "CONT-"
    + df_prestamos_tra["prestamo_id"]
        .astype(str)
        .str.zfill(8)
)

df_prestamos_tra[df_prestamos_tra.numero_contrato != num_pag_gene]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


## fecha_otorgamiento 

In [22]:
# Transforma a formato fecha y cambia las fechas a nan, en caso error 
error_fecha_otor = pd.to_datetime(
    df_prestamos_tra.fecha_otorgamiento,
    errors='coerce'
)
# Muestra las fechas que generan error 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra.fecha_otorgamiento[error_fecha_otor.isna()]

1       2021/02/12
3       19/04/2024
4       19-04-2020
7       2024/07/07
8       20-02-2020
           ...    
6492    13/10/2022
6494    2023/08/19
6495    2021/02/13
6496    20/11/2021
6497    28/08/2023
Name: fecha_otorgamiento, Length: 4232, dtype: object

In [23]:
# Trasforma a formato correcto, utilizando format = 'mixed'
df_prestamos_tra["fecha_otorgamiento"] = pd.to_datetime(
    df_prestamos_tra.fecha_otorgamiento,
    errors='coerce',
    format='mixed',
    dayfirst=True
)
# Muestra las fechas que antes generaban error 
df_prestamos_tra.fecha_otorgamiento[error_fecha_otor.isna()]

1      2021-02-12
3      2024-04-19
4      2020-04-19
7      2024-07-07
8      2020-02-20
          ...    
6492   2022-10-13
6494   2023-08-19
6495   2021-02-13
6496   2021-11-20
6497   2023-08-28
Name: fecha_otorgamiento, Length: 4232, dtype: datetime64[ns]

## fecha_vencimiento

In [24]:
# Transforma a formato fecha y cambia las fechas a nan, en caso error 
error_fecha_ven = pd.to_datetime(
    df_prestamos_tra.fecha_vencimiento,
    errors='coerce'
)
# Muestra las fechas que generar error 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra.fecha_vencimiento[error_fecha_ven.isna()]

5       25/09/2045
6       2024/08/18
7       2024/10/05
8       30/01/2024
9       2025/11/23
           ...    
6492    2052/05/08
6494    15-04-2047
6495    27-07-2024
6496    2024/02/08
6498    13/04/2026
Name: fecha_vencimiento, Length: 4249, dtype: object

In [25]:
# Trasforma a formato correcto, utilizando format = 'mixed'
df_prestamos_tra["fecha_vencimiento"] = pd.to_datetime(
    df_prestamos_tra.fecha_vencimiento,
    errors='coerce',
    format='mixed',
    dayfirst=True
)
# Muestra las fechas que antes generaban error 
df_prestamos_tra.fecha_vencimiento[error_fecha_ven.isna()]

5      2045-09-25
6      2024-08-18
7      2024-10-05
8      2024-01-30
9      2025-11-23
          ...    
6492   2052-05-08
6494   2047-04-15
6495   2024-07-27
6496   2024-02-08
6498   2026-04-13
Name: fecha_vencimiento, Length: 4249, dtype: datetime64[ns]

In [26]:
# Muesta las fechas de otorgamiento que sean mayores a las fechas de vencimiento  
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra[["cliente_id","fecha_otorgamiento","fecha_vencimiento"]][
    df_prestamos_tra.fecha_otorgamiento > df_prestamos_tra.fecha_vencimiento
    ]

,cliente_id,fecha_otorgamiento,fecha_vencimiento


## monto_original 

In [27]:
# Verifica que los montos no sean negativos o cero 
# Resultados Esperados: Tabla Vacia
df_prestamos_tra.monto_original[df_prestamos_tra.monto_original <= 0]

21      -2565.89
75      -5090.41
197    -17438.55
235    -13969.34
435     -3788.57
          ...   
6120   -44597.90
6209    -3796.96
6247    -4575.28
6339    -2304.80
6351   -10779.93
Name: monto_original, Length: 63, dtype: float64

In [28]:
# Aplica valor absoluto para estos valores 
df_prestamos_tra["monto_original"] = df_prestamos_tra.monto_original.apply(abs)
# Verifica que los montos no sean negativos o cero 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra.monto_original[df_prestamos_tra.monto_original <= 0]

Series([], Name: monto_original, dtype: float64)

In [29]:
# LEFT JOIN, con productos 
df_merge_veri_monto = df_prestamos_tra.merge(
    right= df_productos_crediticios_tra,
    on="producto_id",
    how='left'
)

# Verifica que los montos se encuentren en el rango de monto correspondiente para dicho producto
df_merge_veri_monto[["producto_id","monto_original","monto_minimo","monto_maximo"]][
                (df_merge_veri_monto.monto_original < df_merge_veri_monto.monto_minimo) 
                | (
                    (df_merge_veri_monto.monto_maximo.notna())
                    & (df_merge_veri_monto.monto_original > df_merge_veri_monto.monto_maximo))
                ]

,producto_id,monto_original,monto_minimo,monto_maximo


## saldo_capital_vigente

In [30]:
# Verifica si los saldos son negativos
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra.saldo_capital_vigente[df_prestamos_tra.saldo_capital_vigente < 0]

1155     -5391.42
1345      -565.35
1414     -2547.99
1566     -1417.95
1739   -143989.36
1760      -521.49
2411   -126320.09
2516     -1057.33
2575     -2422.22
2620     -5884.36
2673   -329230.22
3456    -26880.58
3621     -4940.04
4155     -1870.07
4352     -3082.11
4485    -77463.65
4514      -431.68
4546     -5262.19
4892     -4786.66
4939     -3156.26
4992    -11677.26
5320    -98739.06
5354    -26620.31
5368      -977.28
5683    -33283.97
5984      -674.34
6070     -1382.43
6084    -11187.10
6101    -12894.13
Name: saldo_capital_vigente, dtype: float64

In [31]:
# Aplica valor absoluto
df_prestamos_tra["saldo_capital_vigente"] = df_prestamos_tra.saldo_capital_vigente.apply(abs)
# Verifica si los saldos son negativos
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra.saldo_capital_vigente[df_prestamos_tra.saldo_capital_vigente < 0]

Series([], Name: saldo_capital_vigente, dtype: float64)

## tasa_interes_nominal_anual

In [32]:
# Verifica si la tasa_interes_nominal_anual es negativa o 0 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra.tasa_interes_nominal_anual[df_prestamos_tra.tasa_interes_nominal_anual <= 0]

74     -5.0
452     0.0
481     0.0
853     0.0
1091    0.0
1354   -5.0
1385    0.0
1956    0.0
2095   -5.0
2269   -5.0
2580    0.0
3099   -5.0
3493    0.0
4109   -5.0
4296    0.0
4567   -5.0
5242    0.0
5268   -5.0
5321   -5.0
5565    0.0
6066    0.0
Name: tasa_interes_nominal_anual, dtype: float64

In [33]:
# Aplica valor absoluto 
df_prestamos_tra["tasa_interes_nominal_anual"] = df_prestamos_tra.tasa_interes_nominal_anual.apply(abs)

# Por el momento tovia no arreglo la tasa de interes nominal, porque, puedo cambiar estos valores tomando en cuenta las tasas efectivas 
# Verifica si la tasa_interes_nominal_anual es negativa o 0 
# Resultados Esperados: Tabla Vacia
df_prestamos_tra.tasa_interes_nominal_anual[df_prestamos_tra.tasa_interes_nominal_anual <= 0]

452     0.0
481     0.0
853     0.0
1091    0.0
1385    0.0
1956    0.0
2580    0.0
3493    0.0
4296    0.0
5242    0.0
5565    0.0
6066    0.0
Name: tasa_interes_nominal_anual, dtype: float64

## tasa_interes_efectiva_anual

In [34]:
# Verifica si existen tasas efectivas anuales negativas o igual a 0
# Resultado Esperado: Tabla Vacia 
df_prestamos_tra[df_prestamos_tra.tasa_interes_efectiva_anual <= 0]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


In [35]:
# Aplica valor absoluto 
df_prestamos_tra["tasa_interes_efectiva_anual"] = df_prestamos_tra.tasa_interes_efectiva_anual.apply(abs)
# Verifica si existen tasas efectivas anuales negativas o igual a 0
# Resultado Esperado: Tabla Vacia 
df_prestamos_tra[df_prestamos_tra.tasa_interes_efectiva_anual <= 0]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


## Proceso de Revision de Tasas (Tasa nominal y efectiva)

In [36]:
# LEFT JOIN, producto_id 
df_merge_veri_tasas = df_prestamos_tra.merge(
    right=df_productos_crediticios_tra,
    on="producto_id",
    how="left"
)

# Verifica que la tasa nominal este dentro del rango de ese producto 
# Resultados Esperados: Tabla Vacia
error_tasa_nominal = df_merge_veri_tasas[["prestamo_id","tasa_interes_nominal_anual","tasa_interes_efectiva_anual","tasa_nom_min","tasa_nom_max"]][
    (df_merge_veri_tasas.tasa_interes_nominal_anual < df_merge_veri_tasas.tasa_nom_min)
    | (df_merge_veri_tasas.tasa_interes_nominal_anual > df_merge_veri_tasas.tasa_nom_max)
].copy()
error_tasa_nominal

,prestamo_id,tasa_interes_nominal_anual,tasa_interes_efectiva_anual,tasa_nom_min,tasa_nom_max
74,75,5.0,30.3880,18.0,36.0
452,453,0.0,60.1032,20.0,48.0
481,482,0.0,31.3478,24.0,42.0
853,854,0.0,11.5719,7.5,11.0
1091,1092,0.0,57.4299,20.0,48.0
1331,1332,250.0,17.2849,2.5,18.0
1354,1355,5.0,21.3049,18.0,36.0
1385,1386,0.0,43.1724,24.0,42.0
1919,1920,250.0,25.7097,12.0,24.0
1956,1957,0.0,37.7121,18.0,36.0


In [37]:
# Recalcula la tasa nominal usando la columna tasa_interes_efectiva_anual considerando una capitalización mensual
error_tasa_nominal["nueva_tasa_nominal"] = error_tasa_nominal.tasa_interes_efectiva_anual.apply(
    lambda tasa_efectiva: 
        round(   
            (12 * 100*(
                (1 + tasa_efectiva/100)**(1/12)- 1
                )
            ),
            4
        )
        if pd.notna(tasa_efectiva)
        else np.nan
)

# Verifica si la nueva tasa nominal respeta los limites de la tabla clientes
# Resultados Esperados: Tabla Vacia
error_tasa_nominal[
    (error_tasa_nominal.nueva_tasa_nominal < error_tasa_nominal.tasa_nom_min)
    | (error_tasa_nominal.nueva_tasa_nominal > error_tasa_nominal.tasa_nom_max)
]

,prestamo_id,tasa_interes_nominal_anual,tasa_interes_efectiva_anual,tasa_nom_min,tasa_nom_max,nueva_tasa_nominal


In [38]:
# Reemplaza los valores antiguos con las nuevas tasas nominales recalculadas 
df_prestamos_tra.loc[error_tasa_nominal.index,"tasa_interes_nominal_anual"] = error_tasa_nominal.nueva_tasa_nominal

# Revisa que los cambios 
df_prestamos_tra.loc[error_tasa_nominal.index,["prestamo_id","tasa_interes_nominal_anual"]]

,prestamo_id,tasa_interes_nominal_anual
74,75,26.83
452,453,48.00
481,482,27.58
853,854,11.00
1091,1092,46.25
1331,1332,16.05
1354,1355,19.47
1385,1386,36.43
1919,1920,23.10
1956,1957,32.43


In [39]:
#==========================================================================================================
# VERIFICA QUE EL CAMBIO ES IGUAL DE AMBOS SENTIDOS, ES DECIR DE EFECTIVA A NOMINAL Y DE NOMINAL A EFECTIVA
#==========================================================================================================

# Selecciona las columnas a utilizar 
tasa_efec = df_prestamos_tra[['prestamo_id','producto_id','tasa_interes_nominal_anual','tasa_interes_efectiva_anual']].copy()

# Recalcula las tasas efectivas utilizando las tasas nominales
tasa_efec["efec_anual_recal"] = tasa_efec.tasa_interes_nominal_anual.apply(
    lambda inte_nominal: 
        round(
            (  
                (
                    (1+(inte_nominal/(12*100))
                    )**12 - 1
                )*100
            ),
            4
        ) 
        if pd.notna(inte_nominal)
        else np.nan
    )

# Realiza un join, con los productos crediticios pero solo con los valores que no coinciden
verificando_tasa_efecti = tasa_efec.merge(
    right=df_productos_crediticios_tra,
    on="producto_id",
    how="left"
)

# Recalcula la tasa nominal utilizando ahora la tasa efectiva existente 
verificando_tasa_efecti["nomi_anual_recal"] = verificando_tasa_efecti.tasa_interes_efectiva_anual.apply(
    lambda tasa_efectiva: 
        round(   
            (12 * 100*(
                (1 + tasa_efectiva/100)**(1/12)- 1
                )
            ),
            4
        )
        if pd.notna(tasa_efectiva)
        else np.nan
)

# Verifica si los recalculos de los valores son correctos, se agrega +- 0.03 por diferencias en el redondeo
# Resultados Esperados: Tabla Vacia
verificando_tasa_efecti[
    ~(verificando_tasa_efecti.tasa_interes_nominal_anual.between(
        left= verificando_tasa_efecti.nomi_anual_recal - 0.03,
        right= verificando_tasa_efecti.nomi_anual_recal +0.03,
        inclusive = 'both'
        )
    )
    |
    ~(verificando_tasa_efecti.tasa_interes_efectiva_anual.between(
        left= verificando_tasa_efecti.efec_anual_recal - 0.03,
        right= verificando_tasa_efecti.efec_anual_recal + 0.03,
        inclusive = 'both'
        )
    )
]

,prestamo_id,producto_id,tasa_interes_nominal_anual,tasa_interes_efectiva_anual,efec_anual_recal,nombre_producto,tipo_credito,tasa_nom_min,tasa_nom_max,plazo_min_meses,plazo_max_meses,monto_minimo,monto_maximo,requiere_garantia,moneda,nomi_anual_recal
2269,2270,3,5.0,18.3775,5.1162,Crédito Vehicular,Vehicular,2.5,18.0,12,72,10000.0,200000.0,True,PEN,16.99
4109,4110,3,5.0,19.5618,5.1162,Crédito Vehicular,Vehicular,2.5,18.0,12,72,10000.0,200000.0,True,PEN,18.00


## plazo_meses

In [40]:
# Verifica si existen plazos de meses negativos o 0
# Resultados Esperados: Tabla Vacia
df_prestamos_tra.plazo_meses[df_prestamos_tra.plazo_meses <= 0]

Series([], Name: plazo_meses, dtype: int64)

In [41]:
# Por el momento solo se va a quedar como esta, porque tendre que comprar si el plazo meses es igual al número de cuotas y la frecuencia de pago. 
# Muestra cuantos registros no coinciden
df_plazo_meses_revi = df_prestamos_tra[["prestamo_id","fecha_otorgamiento","fecha_vencimiento","plazo_meses"]].copy()
df_plazo_meses_revi["diferencia_meses"] = (df_plazo_meses_revi.fecha_vencimiento.dt.year- df_plazo_meses_revi.fecha_otorgamiento.dt.year)*12 + (df_plazo_meses_revi.fecha_vencimiento.dt.month - df_plazo_meses_revi.fecha_otorgamiento.dt.month) 

df_plazo_meses_revi[df_plazo_meses_revi.diferencia_meses != df_plazo_meses_revi.plazo_meses]

,prestamo_id,fecha_otorgamiento,fecha_vencimiento,plazo_meses,diferencia_meses
0,1,2023-05-07,2025-10-23,30,29
1,2,2021-02-12,2025-07-21,54,53
5,6,2020-08-07,2045-09-25,306,301
8,9,2020-02-20,2024-01-30,48,47
9,10,2021-06-17,2025-11-23,54,53
...,...,...,...,...,...
6490,6491,2021-02-08,2023-01-29,24,23
6492,6493,2022-10-13,2052-05-08,360,355
6494,6495,2023-08-19,2047-04-15,288,284
6495,4165,2021-02-13,2024-07-27,42,41


## tipo_credito

In [42]:
# Verifica el formato del tipo del credito 
# Resultados Esperados: 'Personal', 'Microempresa', 'Vehicular', 'Hipotecario', 'Consumo'
df_prestamos_tra.tipo_credito.unique()

array(['Personal', 'Microempresa', 'Hipotecario', 'Vehicular', 'PeRsOnAl',
       'Consumo', 'Microempresa ', 'CoNsUmO', 'consumo', ' PerSonAl',
       'Consumo  ', 'VehIcuLar', 'PerSonAl ', '  MicRoeMprEsa',
       'VeHiCuLaR', '  Consumo', 'MicRoeMprEsa', '  PERSONAL',
       'microempresa', '  PerSonAl', 'MiCrOeMpReSa', ' VehIcuLar',
       'Consumo ', ' Microempresa', '  VeHiCuLaR', ' microempresa',
       'CONSUMO  ', 'MICROEMPRESA', ' MICROEMPRESA', '  Microempresa',
       '  Personal', 'MicRoeMprEsa  ', ' CONSUMO', 'hipotecario',
       ' personal', 'PERSONAL', '  PeRsOnAl', 'VEHICULAR', 'hipotecario ',
       '  consumo', 'PERSONAL  ', 'CONSUMO', '  microempresa',
       'MicRoeMprEsa ', ' Personal', 'CoNsUmO  ', 'vehicular', 'PerSonAl',
       '  CONSUMO', 'Microempresa  ', 'MiCrOeMpReSa ', 'Personal  ',
       '  Vehicular', ' PERSONAL', ' consumo', 'personal', ' Vehicular',
       'MICROEMPRESA ', 'ConSumO', ' MicRoeMprEsa', 'PeRsOnAl  ',
       'personal  ', '  MICROEMPRES

In [43]:
# Aplica el formato correspondiente 
df_prestamos_tra["tipo_credito"] = df_prestamos_tra.tipo_credito.apply(limpiar_texto)
# Resultados Esperados: 'Personal', 'Microempresa', 'Vehicular', 'Hipotecario', 'Consumo'
df_prestamos_tra.tipo_credito.unique()

array(['Personal', 'Microempresa', 'Hipotecario', 'Vehicular', 'Consumo'],
      dtype=object)

In [44]:
# verifica que el tipo de prestamo coincida con el tipo de credito de productos crediticios 
# Resultados Esperados: Tabla Vacia 
verificar_tipo_credito = df_prestamos_tra.merge(
    right=df_productos_crediticios_tra,
    on='producto_id',
    how='left',
    suffixes=["_pres","_produc"]
)

verificar_tipo_credito[["producto_id","tipo_credito_pres","tipo_credito_produc"]][
    verificar_tipo_credito.tipo_credito_pres != verificar_tipo_credito.tipo_credito_produc
    ]

,producto_id,tipo_credito_pres,tipo_credito_produc


## moneda

In [45]:
# Verifica el formato de la modena 
# Resultados Esperados: 'PEN','USD','n/a'
df_prestamos_tra.moneda.unique()

array(['Sol', 'PEN', 's/.', 'pen', 'USD', 'PEN ', 'Soles', 'USD ', 'usd'],
      dtype=object)

In [46]:
# En lugar de limpiar y verificar que coincidan voy a reemplazar de manera directa con los datos de la tabla productos_crediticios 

# LEFT JOIN, producto_id 
df_merge_limpiar_moneda = df_prestamos_tra.merge(
    right=df_productos_crediticios_tra,
    on='producto_id',
    how='left',
    suffixes=["_pres","_produc"]
)

df_prestamos_tra["moneda"] = df_merge_limpiar_moneda.moneda_produc
df_prestamos_tra.moneda.unique()

array(['PEN', 'USD'], dtype=object)

## frecuencia_pago

In [47]:
# Resultados Esperados: 'Mensual', 'n/a'
df_prestamos_tra.frecuencia_pago.unique()

array(['Mensual', 'mensual', 'Mensual ', 'MENSUAL'], dtype=object)

In [48]:
# Aplica el formato adecuado para la columna
df_prestamos_tra["frecuencia_pago"] = df_prestamos_tra.frecuencia_pago.apply(limpiar_texto)
# Resultados Esperados: 'Mensual', 'n/a'
df_prestamos_tra.frecuencia_pago.unique()

array(['Mensual'], dtype=object)

## cuota_programada

**Nota**: La verificacion para saber si las cuotas son correctas se realiza en:   
[Revision Cuota Programada](#proceso-de-revision-cuota_programada)

In [49]:
# Verifica si existe número negativos o 0 
# Resultado Esperado: Tabla Vacia  
df_prestamos_tra.cuota_programada[df_prestamos_tra.cuota_programada <= 0]

Series([], Name: cuota_programada, dtype: float64)

## numero_cuotas_total

In [50]:
# Verifica si las cuotas totales son negativas o 0 
# Resultado Esperado: Tabla Vacia
df_prestamos_tra[df_prestamos_tra.numero_cuotas_total <= 0 ]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


## numero_cuotas_pagadas

In [51]:
# Verifica si las cuotas pagadas son negativas o 0 
# Resultado Esperado: Tabla Vacia
df_prestamos_tra[df_prestamos_tra.numero_cuotas_pagadas <= 0 ]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


In [52]:
# Verifica si las cuotas pagadas son negativas, 0 o si es mayor a la cantidad total de cuotas
# Resultado Esperado: Tabla Vacia 
error_cuotas_pagadas = df_prestamos_tra[["prestamo_id","numero_cuotas_total","numero_cuotas_pagadas"]][
                (df_prestamos_tra.numero_cuotas_pagadas <= 0) 
                | (df_prestamos_tra.numero_cuotas_pagadas >df_prestamos_tra.numero_cuotas_total)].copy()
error_cuotas_pagadas

,prestamo_id,numero_cuotas_total,numero_cuotas_pagadas
193,194,60,65
333,334,36,44
340,341,15,24
447,448,21,29
494,495,12,15
...,...,...,...
6009,6010,15,19
6235,6236,48,56
6347,6348,33,40
6429,6430,6,11


## numero_cuotas_pendientes

In [53]:
# Verifica si las cuotas pendientes son negativas o si son mayores las cuotas totales 
# Resultado Esperado: Tabla Vacia 
df_prestamos_tra[["prestamo_id","numero_cuotas_total","numero_cuotas_pendientes"]][
                (df_prestamos_tra.numero_cuotas_pendientes < 0) 
                | (df_prestamos_tra.numero_cuotas_pendientes >df_prestamos_tra.numero_cuotas_total)].head()

,prestamo_id,numero_cuotas_total,numero_cuotas_pendientes


## Verificar Cuotas (totales, pagas y pendientes)

Revisando mejor los datos, puedo ver que todos los registros de plazo_meses son iguales al numero_cuotas_total, por eso los puedo considerar como columnas ancla y en base a esos datos, puedo calcular las cuotas pagadas y pendientes.

In [54]:
# Verifica si el numero de cuotas es igual al plazo en meses 
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra[df_prestamos_tra.plazo_meses != df_prestamos_tra.numero_cuotas_total]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


**Nota**: Se realizara otra revision luego de limpiar la tabla pagos, porque, tiene que coincidir el numero de cuotas pagadas con las cuotas registradas en pagos, la limpieza se realizara en **08_tabla_bronce.pagos.ipynb**. 

In [55]:
# Verifica si el numero de cuotas_totales no coincide con las cuotas pagadas y las cuotas pendientes  
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra[df_prestamos_tra.numero_cuotas_total != df_prestamos_tra.numero_cuotas_pagadas + df_prestamos_tra.numero_cuotas_pendientes]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real
193,194,947,14,1,28,CONT-00000194,2024-08-23,2029-07-28,10247.81,9831.11,...,56,Vigente,0,Normal,Carta Fianza,0.0,Emergencia Médica,Cheque de Gerencia,22-09-2024,21/12/2024
333,334,2012,23,1,37,CONT-00000334,2022-11-12,2025-10-27,11543.12,4469.12,...,10,Vigente,0,Normal,Aval,0.0,Emergencia Médica,Transferencia Bancaria,2022/12/12,2024-12-31
340,341,2513,15,5,32,CONT-00000341,2020-06-04,2021-08-28,1627.91,0.00,...,0,Cancelado,0,Normal,Aval,0.0,Muebles y Enseres,TrAnSfErEnCiA BaNcArIa,2020/07/04,2021-08-28
447,448,2804,15,8,32,CONT-00000448,2023-02-15,2024-11-06,11418.51,0.00,...,0,Cancelado,0,Normal,Sin Garantía,0.0,Compra de Inventario,Agencia,17-03-2023,2024/11/06
494,495,3484,15,1,51,CONT-00000495,2023-01-22,2024-01-17,12120.62,0.00,...,0,Cancelado,0,Normal,Carta Fianza,0.0,Libre Disponibilidad,Agencia,2023/02/21,2024-01-17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6009,6010,3238,15,5,51,CONT-00006010,2023-03-23,2024-06-15,5106.66,0.00,...,0,Cancelado,0,Normal,Carta Fianza,0.0,Muebles y Enseres,CHEQUE DE GERENCIA,2023-04-22,2024-06-15
6235,6236,2952,24,1,18,CONT-00006236,2024-09-20,2028-08-30,3711.52,3577.68,...,45,Vigente,0,Normal,Aval,0.0,Viaje / Turismo,TrAnSfErEnCiA BaNcArIa,20/10/2024,2024/12/19
6347,6348,825,4,8,19,CONT-00006348,2023-04-18,2026-01-02,18890.41,8822.35,...,13,Vigente,0,Normal,Aval,0.0,Adquisición de Activo Fijo,Agencia,18-05-2023,2024/12/08
6429,6430,3013,12,1,42,CONT-00006430,2023-10-29,2024-04-26,2246.60,821.84,...,0,Moroso,278,Dudoso,Carta Fianza,0.0,Libre Disponibilidad,Transferencia Bancaria,2023-11-28,26-04-2024


Primero voy arreglar las cuotas que se que estan mal, en este caso son las cuotas pagadas que son mayores al numero de cuotas totales

In [56]:
# Arregla cuotas pagas, cuotas pagadas = cuotas_totales - cuotas_pendientes
df_prestamos_tra.loc[error_cuotas_pagadas.index,"numero_cuotas_pagadas"] = df_prestamos_tra.loc[error_cuotas_pagadas.index,"numero_cuotas_total"] - df_prestamos_tra.loc[error_cuotas_pagadas.index,"numero_cuotas_pendientes"]
# Muestra los registros que antes generaban error
df_prestamos_tra.loc[error_cuotas_pagadas.index,["prestamo_id","numero_cuotas_total","numero_cuotas_pagadas","numero_cuotas_pendientes"]]

,prestamo_id,numero_cuotas_total,numero_cuotas_pagadas,numero_cuotas_pendientes
193,194,60,4,56
333,334,36,26,10
340,341,15,15,0
447,448,21,21,0
494,495,12,12,0
...,...,...,...,...
6009,6010,15,15,0
6235,6236,48,3,45
6347,6348,33,20,13
6429,6430,6,6,0


In [57]:
# Verifica si siguen existiendo datos erroneos
# Resultados Esperados: Tabla Vacia
df_prestamos_tra[df_prestamos_tra.numero_cuotas_total != df_prestamos_tra.numero_cuotas_pagadas + df_prestamos_tra.numero_cuotas_pendientes]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


## estado

**Nota**: Por el momento la limpieza va a ser solo del formato, para validar que realmente el estado sea el correcto, voy a terminar de limpiar todas las columnas restantes, la limpieza se va a realizar en:  
[Revision Estado del Prestamo](#proceso-de-revision-del-estado-del-prestamo)

In [58]:
# Resultados Esperados: 'Vigente', 'Cancelado', 'Moroso', 'Refinanciado', 'Castigado'
df_prestamos_tra.estado.unique()

array(['Vigente', 'Cancelado', 'Moroso', 'Castigado', 'vigente',
       'Refinanciado', 'CANCELADO', 'cancelado', 'VIGENTE', 'Cancelado ',
       'CASTIGADO', 'Vigente ', 'castigado'], dtype=object)

In [59]:
# Arregla espacios en blanco y formatos de texto 
# Resultados Esperados: 'Vigente', 'Cancelado', 'Moroso', 'Refinanciado', 'Castigado'
df_prestamos_tra["estado"] = df_prestamos_tra.estado.apply(limpiar_texto)
df_prestamos_tra.estado.unique()

array(['Vigente', 'Cancelado', 'Moroso', 'Castigado', 'Refinanciado'],
      dtype=object)

##  dias_mora

In [60]:
# Resultados Esperados: Tabla Vacia 
df_prestamos_tra.dias_mora[df_prestamos_tra.dias_mora < 0]

Series([], Name: dias_mora, dtype: int64)

## clasificacion_riesgo_sbs

- **Categoría 0: Normal**  
Créditos de consumo, personal y vehicular: pagos puntuales o hasta 8 días de retraso  
Créditos de microempresa: pagos puntuales o hasta 8 días de retraso  
Créditos hipotecarios: pagos puntuales o hasta 30 días de retraso  
- **Categoría 1: Con problemas potenciales (CPP)**  
Créditos de consumo, personal y vehicular: retrasos entre 9 y 30 días  
Créditos de microempresa: retrasos entre 9 y 30 días  
Créditos hipotecarios: retrasos entre 31 y 60 días  
- **Categoría 2: Deficiente**  
Créditos de consumo, personal y vehicular: retrasos entre 31 y 60 días  
Créditos de microempresa: retrasos entre 31 y 60 días  
Créditos hipotecarios: retrasos entre 61 y 120 días  
- **Categoría 3: Dudoso**  
Créditos de consumo, personal y vehicular: retrasos entre 61 y 120 días  
Créditos de microempresa: retrasos entre 61 y 120 días  
Créditos hipotecarios: retrasos entre 121 y 365 días  
- **Categoría 4: Pérdida**  
Créditos de consumo, personal y vehicular: más de 120 días de atraso  
Créditos de microempresa: más de 120 días de atraso  
Créditos hipotecarios: más de 365 días de atraso  

In [61]:
# Resultados Esperados: 'Normal', 'Dudoso', 'n/a', 'CPP', 'Pérdida', 'Deficiente'
df_prestamos_tra.clasificacion_riesgo_sbs.unique()

array(['Normal', 'Dudoso', None, 'CPP', 'Normal ', 'Pérdida', 'Cpp',
       'Deficiente', 'normal', 'deficiente', 'DUDOSO', 'NORMAL', 'dudoso',
       'cpp', 'PERDIDA', 'DEFICIENTE', 'perdida', 'Pérdida '],
      dtype=object)

In [62]:
# Aplica la clasificacion correspondiente segun el tipo de credito 
df_prestamos_tra["clasificacion_riesgo_sbs"] = df_prestamos_tra.apply(
    axis=1,
    func= lambda x: formato_clasificacion_riesgo_sbs(x['tipo_credito'],x['dias_mora'])
    )
df_prestamos_tra.clasificacion_riesgo_sbs.unique()


array(['Normal', 'Pérdida', 'Deficiente', 'Dudoso', 'CPP'], dtype=object)

## garantia_tipo

In [63]:
# Resultados Esperados: 'n/a', 'Aval', 'Carta Fianza', 'Sin Garantía', 'Bien Inmueble', 'Prenda Vehicular', 'Bien Mueble'
df_prestamos_tra.garantia_tipo.unique()

array([None, 'Aval', 'Carta Fianza', 'Sin Garantía', 'Bien Inmueble',
       'Prenda Vehicular', 'Bien Mueble'], dtype=object)

In [64]:
# Arregla algunos formatos y cambia nan por 'n/a'
df_prestamos_tra["garantia_tipo"] = df_prestamos_tra.garantia_tipo.apply(limpiar_texto)
df_prestamos_tra.garantia_tipo.unique()

array(['n/a', 'Aval', 'Carta Fianza', 'Sin Garantía', 'Bien Inmueble',
       'Prenda Vehicular', 'Bien Mueble'], dtype=object)

**Nota**: Existen registros, que no coinciden pero no puedo arriesgarme a colocar culquier garantia, en su lugar voy a separar estos registros para revisarlos 

In [65]:
# Verifica si existan registros que no tengan garantia, cuando el credito solicita que tenga garantia 
# Resultado Esperado: Tabla Vacia
df_merge_verificar_garantia = df_prestamos_tra.merge(
    right=df_productos_crediticios_tra,
    on='producto_id',
    how='left',
)
# Verifica si existen registros que no tengan garantia apesar de requerir garantia 
df_revisar_garantia = df_merge_verificar_garantia[['prestamo_id','cliente_id','producto_id','garantia_tipo','requiere_garantia']][(df_merge_verificar_garantia.garantia_tipo == 'n/a')
                & (df_merge_verificar_garantia.requiere_garantia == True)].copy()
df_revisar_garantia

,prestamo_id,cliente_id,producto_id,garantia_tipo,requiere_garantia
244,245,1569,2,n/a,True
443,444,3655,2,n/a,True
1447,1448,1488,2,n/a,True
1656,1657,2383,3,n/a,True
1679,1680,1419,3,n/a,True
2216,2217,982,3,n/a,True
2482,2483,2467,3,n/a,True
2586,2587,2267,3,n/a,True
3032,3033,210,2,n/a,True
3507,3508,2178,2,n/a,True


In [66]:
df_revisar_garantia["ERROR"] = "NO TIENEN GARANTIA, A PESAR DE NECESITAR POR EL TIPO DE PRESTAMO"

## garantia_valor

In [67]:
# Verifica que no existan garantias con valor negativo 
# Resultados Esperados: Tabla Vacias 
df_prestamos_tra[df_prestamos_tra.garantia_valor < 0 ]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


In [68]:
# Cuenta la cantidad de prestamos para cada tipo de credito
df_prestamos_tra.garantia_tipo.value_counts().sort_values(ascending=False)

garantia_tipo
Sin Garantía        1761
Carta Fianza        1741
Aval                1731
Prenda Vehicular     390
Bien Mueble          382
Bien Inmueble        365
n/a                  129
Name: count, dtype: int64

- No estoy muy seguro de como actuar en esta situación, porque, no estoy del todo familiarizado con los terminos.   
- Hasta donde pude indagar, Aval y Carta Fianza si pueden tener un valor, pero por lo que veo en estos datos, parece que no, porque el 100% de este tipo de garantias no tiene un valor.  
- En un entorno real estas consultas se tendrian que resolver con el equipo encargado de ver esta prestamos para que explique la logica a seguir.

In [69]:
# Verifica la cantidad de garantias con valor 0 segun el tipo de garantia
df_prestamos_tra[df_prestamos_tra.garantia_valor == 0 ].groupby("garantia_tipo")["garantia_valor"].size().sort_values(ascending=False)

garantia_tipo
Sin Garantía    1761
Carta Fianza    1741
Aval            1731
n/a              114
Name: garantia_valor, dtype: int64

## proposito_credito

In [70]:
#  'Viaje / Turismo', 'Mejoras del Hogar', 'Expansión de Negocio', 'Capital de Trabajo', 'Compra de Vivienda', 'n/a','Compra de Inventario', 'Maquinaria', 'Gastos de Estudios', 'Emergencia Médica', 'Libre Disponibilidad', 'Ampliación y Remodelación', 'Vehículo Nuevo', 'Vehículo Usado', 'Vestimenta y Calzado', 'Electrodomésticos', 'Refinanciamiento Hipotecario', 'Atención Médica', 'Vehículo Comercial', 'Adquisición de Activo Fijo', 'Muebles y Enseres', 'Construcción de Vivienda'
df_prestamos_tra.proposito_credito.unique()

array(['Viaje / Turismo', 'Mejoras del Hogar', 'Expansión de Negocio',
       'Capital de Trabajo', 'Compra de Vivienda', None,
       'Compra de Inventario', 'Maquinaria', 'Gastos de Estudios',
       'Emergencia Médica', 'Libre Disponibilidad',
       'Ampliación y Remodelación', 'Vehículo Nuevo', 'Vehículo Usado',
       'Vestimenta y Calzado', 'Electrodomésticos',
       ' refinanciamiento hipotecario', 'Atención Médica',
       'Refinanciamiento Hipotecario', 'Vehículo Comercial',
       'Adquisición de Activo Fijo', 'EmErGeNcIa mÉdIcA',
       'Muebles y Enseres', '  AmPlIaCiÓn y rEmOdElAcIóN',
       'Emergencia Médica ', 'VEHÍCULO NUEVO', '  MuEbLeS Y EnSeReS',
       'ADQUISICIÓN DE ACTIVO FIJO', 'MeJoRaS DeL HoGaR',
       'EXPANSIÓN DE NEGOCIO', 'Compra de Vivienda ',
       'gastos de estudios ', 'CoMpRa dE ViViEnDa', ' Atención Médica',
       'MAQUINARIA', 'AdqUisIciÓn De ActIvo fiJo', 'COMPRA DE VIVIENDA ',
       'CaPiTaL De tRaBaJo', 'Construcción de Vivienda',
       

In [71]:
# Aplica el formato de texto adecuado
df_prestamos_tra["proposito_credito"] = df_prestamos_tra.proposito_credito.apply(limpiar_texto)

# Resultados Esperados:  'Viaje / Turismo', 'Mejoras del Hogar', 'Expansión de Negocio', 'Capital de Trabajo', 'Compra de Vivienda', 'n/a','Compra de Inventario', 'Maquinaria', 'Gastos de Estudios', 'Emergencia Médica', 'Libre Disponibilidad', 'Ampliación y Remodelación', 'Vehículo Nuevo', 'Vehículo Usado', 'Vestimenta y Calzado', 'Electrodomésticos', 'Refinanciamiento Hipotecario', 'Atención Médica', 'Vehículo Comercial', 'Adquisición de Activo Fijo', 'Muebles y Enseres', 'Construcción de Vivienda'
df_prestamos_tra.proposito_credito.unique()

array(['Viaje / Turismo', 'Mejoras del Hogar', 'Expansión de Negocio',
       'Capital de Trabajo', 'Compra de Vivienda', 'n/a',
       'Compra de Inventario', 'Maquinaria', 'Gastos de Estudios',
       'Emergencia Médica', 'Libre Disponibilidad',
       'Ampliación y Remodelación', 'Vehículo Nuevo', 'Vehículo Usado',
       'Vestimenta y Calzado', 'Electrodomésticos',
       'Refinanciamiento Hipotecario', 'Atención Médica',
       'Vehículo Comercial', 'Adquisición de Activo Fijo',
       'Muebles y Enseres', 'Construcción de Vivienda'], dtype=object)

In [72]:
consumo = [
    "Vestimenta y Calzado",
    "Electrodomésticos",
    "Atención Médica",
    "Muebles y Enseres",
    "n/a"
]

personal = [
    "Gastos de Estudios",
    "Viaje / Turismo",
    "Mejoras del Hogar",
    "Emergencia Médica",
    "Libre Disponibilidad",
    "n/a"
]

microempresa = [
    "Expansión de Negocio",
    "Capital de Trabajo",
    "Compra de Inventario",
    "Adquisición de Activo Fijo",
    "n/a"
]

vehicular = [
    "Maquinaria",
    "Vehículo Nuevo",
    "Vehículo Usado",
    "Vehículo Comercial",
    "n/a"
]

hipotecario = [
    "Ampliación y Remodelación",
    "Compra de Vivienda",
    "Construcción de Vivienda",
    "Refinanciamiento Hipotecario",
    "n/a"
]

In [73]:
# Verifica que el proposito del credito coincida con el tipo_credito
# Resultados Esperados: Tabla Vacia
df_prestamos_tra[['prestamo_id','cliente_id','tipo_credito','proposito_credito']][
    ((df_prestamos_tra.tipo_credito == 'Consumo') & ~(df_prestamos_tra.proposito_credito.isin(consumo)))
    | ((df_prestamos_tra.tipo_credito == 'Personal') & ~(df_prestamos_tra.proposito_credito.isin(personal)))
    | ((df_prestamos_tra.tipo_credito == 'Microempresa') & ~(df_prestamos_tra.proposito_credito.isin(microempresa)))
    | ((df_prestamos_tra.tipo_credito == 'Vehicular') & ~(df_prestamos_tra.proposito_credito.isin(vehicular)))
    | ((df_prestamos_tra.tipo_credito == 'Hipotecario') & ~(df_prestamos_tra.proposito_credito.isin(hipotecario)))
]

,prestamo_id,cliente_id,tipo_credito,proposito_credito


## canal_desembolso

In [74]:
# Resultados Esperados: 'Transferencia Bancaria', 'Agencia', 'Cheque De Gerencia', 'n/a'
df_prestamos_tra.canal_desembolso.unique()

array(['Transferencia Bancaria', 'Agencia', 'Cheque de Gerencia',
       'TRANSFERENCIA BANCARIA', '  Transferencia Bancaria', 'AgEnCiA',
       'Cheque de Gerencia  ', 'CheQue de geRenCia',
       'TraNsfEreNciA bAncAriA', 'transferencia bancaria ',
       'TraNsfEreNciA bAncAriA  ', '  Cheque de Gerencia',
       'ChEqUe dE GeReNcIa ', 'cheque de gerencia',
       'Transferencia Bancaria ', 'TrAnSfErEnCiA BaNcArIa',
       'TrAnSfErEnCiA BaNcArIa  ', '  agencia', ' Transferencia Bancaria',
       'agencia', 'agencia ', ' ChEqUe dE GeReNcIa',
       '  cheque de gerencia', '  TraNsfEreNciA bAncAriA',
       'transferencia bancaria', '  AGENCIA', 'Agencia ', '  Agencia',
       'AgEnCiA  ', ' Cheque de Gerencia', 'ChEqUe dE GeReNcIa',
       'AGENCIA', ' TRANSFERENCIA BANCARIA', ' Agencia', 'AgeNciA',
       ' agencia', 'Cheque de Gerencia ', 'TRANSFERENCIA BANCARIA ',
       'AGENCIA  ', ' AGENCIA', 'CheQue de geRenCia ',
       '  TrAnSfErEnCiA BaNcArIa', 'cheque de gerencia  ', 'Age

In [75]:
# Aplica el formato de texto adecuado 
# Resultados Esperados: 'Transferencia Bancaria', 'Agencia', 'Cheque de Gerencia', 'n/a'
df_prestamos_tra["canal_desembolso"] = df_prestamos_tra.canal_desembolso.apply(limpiar_texto)
df_prestamos_tra.canal_desembolso.unique()

array(['Transferencia Bancaria', 'Agencia', 'Cheque de Gerencia'],
      dtype=object)

## fecha_primer_pago_programado

In [76]:
# Muestra las fechas que generan error 
# Resultados Esperados: Tabla Vacia
error_fecha_primer_pag = pd.to_datetime(
    df_prestamos_tra.fecha_primer_pago_programado, 
    errors='coerce'
)
df_prestamos_tra["fecha_primer_pago_programado"][error_fecha_primer_pag.isna()]

2       22-12-2023
3       19/05/2024
4       19/05/2020
5       2020/09/06
6       27/03/2023
           ...    
6492    2022/11/12
6493    2020/02/06
6496    20/12/2021
6497    2023/09/27
6498    2020/12/10
Name: fecha_primer_pago_programado, Length: 4229, dtype: object

In [77]:
# Cambia el tipo de dato a date y muestra las fechas que antes generaban error
df_prestamos_tra['fecha_primer_pago_programado'] = pd.to_datetime(
    df_prestamos_tra.fecha_primer_pago_programado,
    errors='coerce',
    format='mixed',
    dayfirst=True
)
# Muestra las fechas que generaban error 
# Resultados Esperados: Tabla Vacia
df_prestamos_tra['fecha_primer_pago_programado'][error_fecha_primer_pag.isna()]

2      2023-12-22
3      2024-05-19
4      2020-05-19
5      2020-09-06
6      2023-03-27
          ...    
6492   2022-11-12
6493   2020-02-06
6496   2021-12-20
6497   2023-09-27
6498   2020-12-10
Name: fecha_primer_pago_programado, Length: 4229, dtype: datetime64[ns]

## fecha_ultimo_pago_real

In [78]:
# Muestra las fechas que generar error 
error_fecha_ultimo_pag = pd.to_datetime(
    df_prestamos_tra.fecha_ultimo_pago_real, 
    format='%Y-%m-%d',
    errors='coerce'
)
df_prestamos_tra["fecha_ultimo_pago_real"][error_fecha_ultimo_pag.isna()]

0       27-12-2024
2       16/12/2024
3       2024/10/16
5       14-12-2024
7       2024/10/05
           ...    
6487    20/05/2022
6488    2022/03/08
6490    29-01-2023
6494    2024/12/11
6498    19/12/2024
Name: fecha_ultimo_pago_real, Length: 4206, dtype: object

In [79]:
# Cambia el tipo de dato a date y muestra las fechas que antes generaban error
df_prestamos_tra['fecha_ultimo_pago_real'] = pd.to_datetime(
    df_prestamos_tra.fecha_ultimo_pago_real,
    errors='coerce',
    format='mixed',
    dayfirst=True
)
df_prestamos_tra['fecha_ultimo_pago_real'][error_fecha_ultimo_pag.isna()]

0      2024-12-27
2      2024-12-16
3      2024-10-16
5      2024-12-14
7      2024-10-05
          ...    
6487   2022-05-20
6488   2022-03-08
6490   2023-01-29
6494   2024-12-11
6498   2024-12-19
Name: fecha_ultimo_pago_real, Length: 4206, dtype: datetime64[ns]

In [80]:
# Fechas ultimo pago real futuras 
# Resultados Esperados: Tabla Vacia
df_prestamos_tra[df_prestamos_tra.fecha_ultimo_pago_real.dt.date >= date.today()]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


# Segunda Revision de las columnas

## Proceso de Revision de fecha_primer_pago_programado y fecha_otorgamiento

In [81]:
# Muestra registros donde la fecha de primer pago sea menor a la fecha de otorgamiento del credito
# Resultados Esperados: Tabla Vacia
df_prestamos_tra[['prestamo_id','fecha_otorgamiento','fecha_primer_pago_programado']][
    df_prestamos_tra.fecha_primer_pago_programado < df_prestamos_tra.fecha_otorgamiento
    ]

,prestamo_id,fecha_otorgamiento,fecha_primer_pago_programado


In [82]:
# Los registros siguen esta logica,  fecha_primer_pago_programado == fecha_otorgamiento + 30 dias
# Verifica si esta logica aplica para todos los registros
# Resultados Esperados: Tabla Vacia
df_prestamos_tra[['prestamo_id','fecha_otorgamiento','fecha_primer_pago_programado']][
    df_prestamos_tra.fecha_primer_pago_programado 
    != df_prestamos_tra.fecha_otorgamiento + pd.DateOffset(days=30)
    ]

,prestamo_id,fecha_otorgamiento,fecha_primer_pago_programado


**Nota**: Para arreglar los registros de fecha_ultimo_pago_real, primero tengo que limpiar la tabla pagos, la limpieza se va a realizar en **09_validacion_pagos_prestamos.ipynb**

In [83]:
# Verifica si la fecha de ultimo pago real es menor a la fecha de primer pago programado
# Resultados Esperados: Tabla Vacia
df_prestamos_tra[['prestamo_id','fecha_otorgamiento','fecha_primer_pago_programado','fecha_ultimo_pago_real']][df_prestamos_tra.fecha_ultimo_pago_real <= df_prestamos_tra.fecha_primer_pago_programado]

,prestamo_id,fecha_otorgamiento,fecha_primer_pago_programado,fecha_ultimo_pago_real
1853,1854,2024-12-09,2025-01-08,2024-12-11
4609,4610,2020-12-03,2021-01-02,2020-12-07


## Proceso de Revision fecha_vencimiento 

In [84]:
# Como ya se verifico que fecha_otorgamiento es correcta, tambien que plazo_meses y numero_cuotas_total son correctas puedo verificar si fecha vencimiento es correcta
# En este caso el calculo de fecha_vencimiento no es sumando los meses, es multiplicando 30 por el numero de meses,  fecha_vencimiento = fecha_otorgamiento + 30*(plazo_meses) 

# Recalcula las fechas de vencimiento
fecha_vencimiento_recal = df_prestamos_tra.apply(
    axis = 1,
    func = lambda x: x['fecha_otorgamiento'] + pd.DateOffset(days=30*x['plazo_meses'])
)
# Compara los registros que no cumplan con la logica de fecha vencimiento 
error_fecha_vencimiento_v2 = df_prestamos_tra[['prestamo_id','fecha_otorgamiento','fecha_primer_pago_programado','fecha_vencimiento','plazo_meses']][
    df_prestamos_tra.fecha_vencimiento != fecha_vencimiento_recal
    ]
error_fecha_vencimiento_v2

,prestamo_id,fecha_otorgamiento,fecha_primer_pago_programado,fecha_vencimiento,plazo_meses
78,79,2020-07-12,2020-08-11,2026-05-10,66
420,421,2021-05-13,2021-06-12,2022-06-08,15
531,532,2022-07-04,2022-08-03,2023-12-28,21
617,618,2020-12-06,2021-01-05,2021-04-06,6
633,634,2022-03-19,2022-04-18,2023-09-12,21
717,718,2020-10-01,2020-10-31,2024-12-14,60
813,814,2022-11-05,2022-12-05,2023-11-02,18
916,917,2023-11-13,2023-12-13,2024-07-11,12
936,937,2021-10-24,2021-11-23,2025-03-10,48
1560,1561,2022-12-27,2023-01-26,2026-06-12,48


In [85]:
# Cambia los registros 
df_prestamos_tra['fecha_vencimiento'] = fecha_vencimiento_recal

# Muestra los cambios en la fecha de vencimiento 
df_prestamos_tra.loc[error_fecha_vencimiento_v2.index,['prestamo_id','fecha_otorgamiento','fecha_primer_pago_programado','fecha_vencimiento','plazo_meses']] 

,prestamo_id,fecha_otorgamiento,fecha_primer_pago_programado,fecha_vencimiento,plazo_meses
78,79,2020-07-12,2020-08-11,2025-12-13,66
420,421,2021-05-13,2021-06-12,2022-08-06,15
531,532,2022-07-04,2022-08-03,2024-03-25,21
617,618,2020-12-06,2021-01-05,2021-06-04,6
633,634,2022-03-19,2022-04-18,2023-12-09,21
717,718,2020-10-01,2020-10-31,2025-09-05,60
813,814,2022-11-05,2022-12-05,2024-04-28,18
916,917,2023-11-13,2023-12-13,2024-11-07,12
936,937,2021-10-24,2021-11-23,2025-10-03,48
1560,1561,2022-12-27,2023-01-26,2026-12-06,48


## Proceso de Revision cuota_programada

Para este proyecto se utiliza el sistema frances de creditos, por eso se va a utilizar este sistema para validar que todas las cuota_programada sean validas, la razon, por la que esta verificacion se realiza al final, es porque necesitaba tener, el interes y el numero de periodos. 

prestamo = cuota/TEM * ((1-(1/TEM)**NR_MESES)/(1-(1/TEM)))  
cuota = (prestamo * TEM * (1-(1/TEM))) / (1-(1/TEM)**NR_MESES)  
**Para fines practicos vamos a considerar aceptables todos los valores con margenes de error +-0.03, porque es posible que existan valores diferentes por el momento donde se redondea**

In [86]:
# Selecciona las columnas importantes
verfi_cuota_programada = df_prestamos_tra[['prestamo_id','monto_original','cuota_programada','tasa_interes_efectiva_anual','numero_cuotas_total']].copy()

# Cambia la tasa anual a mensual 
verfi_cuota_programada['tasa_interes_efectiva_mensual'] = verfi_cuota_programada.tasa_interes_efectiva_anual.apply(
    lambda x: ((1+x/100)**(1/12)-1)*100
)

# Recalcula nuevamente la cuota_programada 
verfi_cuota_programada['recal_cuota_programada'] = verfi_cuota_programada.apply(
    axis = 1,
    func= lambda x: 
        round(
            (x['monto_original'] *(1+x['tasa_interes_efectiva_mensual']/100)
                * (1-(1/(1+x['tasa_interes_efectiva_mensual']/100))))
            / (1-(1/(1+x['tasa_interes_efectiva_mensual']/100))**x['numero_cuotas_total']),
            2)
)
# Verifica si existen diferencias entre el recalculo y el valor original
# Resultados Esperados: Tabla Vacia
verfi_cuota_programada[~(verfi_cuota_programada.cuota_programada.between(
    left=verfi_cuota_programada.recal_cuota_programada-0.03,
    right=verfi_cuota_programada.recal_cuota_programada+0.03,
    inclusive='both'
    ))
]

,prestamo_id,monto_original,cuota_programada,tasa_interes_efectiva_anual,numero_cuotas_total,tasa_interes_efectiva_mensual,recal_cuota_programada


## Proceso de Revision de Tasas V2 (tasa_interes_nominal_anual y tasa_interes_efectiva_anual)

Ahora que ya se ha revisado cuota_programada, puedo verificar que el valor de tasa efectiva es correcta, por ello ahora voy arreglar las tasas nominales que antes no coincidian.  
Para entender porque hago esto ahora se puede revisar:  
[Proceso de Revision de Tasas (Tasa nominal y efectiva)](##Proceso-de-Revision-de-Tasas-(Tasa-nominal-y-efectiva))

In [87]:
# Registros incorrectos
verificando_tasa_efecti[["prestamo_id","producto_id","tasa_interes_nominal_anual","nomi_anual_recal","tasa_interes_efectiva_anual","efec_anual_recal","tasa_nom_min","tasa_nom_max"]]

,prestamo_id,producto_id,tasa_interes_nominal_anual,nomi_anual_recal,tasa_interes_efectiva_anual,efec_anual_recal,tasa_nom_min,tasa_nom_max
0,1,6,20.14,20.14,22.1071,22.1071,12.0,24.0
1,2,1,21.56,21.56,23.8234,23.8234,18.0,36.0
2,3,1,36.00,36.00,42.5761,42.5761,18.0,36.0
3,4,4,42.35,42.35,51.6186,51.6186,20.0,48.0
4,5,4,32.20,32.20,37.4040,37.4040,20.0,48.0
...,...,...,...,...,...,...,...,...
6494,6495,2,10.84,10.84,11.3951,11.3951,7.5,11.0
6495,4165,1,35.53,35.53,41.9269,41.9269,18.0,36.0
6496,872,5,32.51,32.51,37.8194,37.8194,24.0,42.0
6497,202,5,42.00,42.00,51.1069,51.1069,24.0,42.0


In [88]:
# Modifiacion manual (ya se valido que la tasa efectiva es la correcta, por eso se usa el recalculo de la tasa efectiva a nominal)
df_prestamos_tra.loc[df_prestamos_tra.prestamo_id == 2270,'tasa_interes_nominal_anual'] = 16.99
df_prestamos_tra.loc[df_prestamos_tra.prestamo_id == 4110,'tasa_interes_nominal_anual'] = 18.00

# Verifica los cambios
df_prestamos_tra.loc[df_prestamos_tra.prestamo_id.isin([2270,4110]),'tasa_interes_nominal_anual']

2269    16.99
4109    18.00
Name: tasa_interes_nominal_anual, dtype: float64

In [89]:
# Verifica nuevamente si las tasas se encuentran dentro de los rangos correspondientes 
# LEFT JOIN, producto_id 
df_merge_veri_tasas_v2 = df_prestamos_tra.merge(
    right=df_productos_crediticios_tra,
    on="producto_id",
    how="left"
)

# Verifica que la tasa nominal este dentro del rango de ese producto 
# Resultados Esperados: Tabla Vacia
df_merge_veri_tasas_v2[["prestamo_id","tasa_interes_nominal_anual","tasa_interes_efectiva_anual","tasa_nom_min","tasa_nom_max"]][
    (df_merge_veri_tasas_v2.tasa_interes_nominal_anual < df_merge_veri_tasas_v2.tasa_nom_min)
    | (df_merge_veri_tasas_v2.tasa_interes_nominal_anual > df_merge_veri_tasas_v2.tasa_nom_max)
]

,prestamo_id,tasa_interes_nominal_anual,tasa_interes_efectiva_anual,tasa_nom_min,tasa_nom_max


## Proceso de Revision de saldo_capital_vigente y numero_cuotas_pendientes

In [90]:
# Verifica si para un prestamo cancelado todavía existen cuotas pendientes
# Resultados Esperados: Tabla Vacia
df_prestamos_tra[(df_prestamos_tra.saldo_capital_vigente == 0) & (df_prestamos_tra.numero_cuotas_pendientes != 0)]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


## Proceso de Revision del estado del prestamo

Patrones para idenficar el estado del prestamo:  
- **Cancelado**: Siempre que no tenga cuotas pendientes no importa si tiene dias mora.  
- **Vigente**: Siempre que tenga cuotas pendientes y no tenga dias_mora. 
- **Refinanciado**: No encuentro un patron para este estado, porque, no importa si se tiene cuotas o si tiene dias_mora, me imagino que es, porque, el refinanciamiento es algo que decide realizar el cliente no tanto una clasificacion directa del banco, por precaucion, no voy a modificar los estados de los prestamos que tengan estado refinanciado. 
- **Moroso**: Siempre que tenga dias_mora menor a la cantidad necesaria para considerarse **castigado** y tenga cuotas pendientes.  
- **Castigado**: Siempre que tenga días_mora mayor o igual a la cantidad necesaria para considerarse **castigado** y tenga cuotas pendientes.
- **Inconsistente**: Es un tipo de estado que voy asignar cuando no se cumplan ninguna de las otras alternativas posibles.  

In [91]:
df_prestamos_tra.estado.unique()

array(['Vigente', 'Cancelado', 'Moroso', 'Castigado', 'Refinanciado'],
      dtype=object)

In [92]:
df_prestamos_tra["estado"] = df_prestamos_tra.apply(
    axis = 1,
    func = lambda x: asignacion_estado_prestamo(x["numero_cuotas_pendientes"],x["dias_mora"],x["estado"],x["tipo_credito"])
)
df_prestamos_tra.estado.value_counts()

estado
Cancelado       3391
Vigente         2428
Castigado        330
Moroso           191
Refinanciado     159
Name: count, dtype: int64

## Proceso de Revision fecha_otorgamiento V2

- En un entorno real, no puedo cambiar tan facilmente las fechas, porque, no estoy seguro, que fecha es la correcta, no me podria arriesgar a modificar ninguna de las dos fechas, en su lugar tendria que buscar informacion o registros anteriores para poder tomar una decision.   
- Como en este proyecto busco mostrar mis capacidades, en lugar de solo separar y aislar estos datos voy a modificar clientes y oficiales, en el archivo 07_validacion_clientes_prestamos. 
- Puede que sea un poco mas complejo, porque, implica modificar nuevamente datos de clientes pero, considero que es necesario.  

### fecha_registro (relacion con clientes)

In [93]:
# LEFT JOIN, por cliente_id 
df_veri_fecha_presta = df_prestamos_tra[['prestamo_id','cliente_id','fecha_otorgamiento']].copy()
df_veri_fecha_clie = df_clientes_tra[['cliente_id','fecha_registro']]
df_merge_verificar_fecha_otor = df_veri_fecha_presta.merge(
    right= df_veri_fecha_clie,
    on="cliente_id",
    how='left'
)
# Verifica que la fecha_otorgamiento no sea mayor a la fecha_registro 
# Resultados Esperados: Tabla Vacia 
df_revisar_fecha_otor = df_merge_verificar_fecha_otor[
                                        (df_merge_verificar_fecha_otor.fecha_otorgamiento 
                                            < df_merge_verificar_fecha_otor.fecha_registro)
                                        ].copy()
df_revisar_fecha_otor

,prestamo_id,cliente_id,fecha_otorgamiento,fecha_registro
277,278,1362,2020-01-08,2032-07-20
1964,1965,1362,2024-01-17,2032-07-20


In [94]:
df_revisar_fecha_otor["ERROR"] = "FECHA DE OTORGAMIENTO DE CREDITO, MENOR A LA FECHA DE REGISTRO DEL CLIENTE" 

### fecha_apertura (relacion con sucursal)

In [95]:
# LEFT JOIN, por sucursal_id 
df_veri_fecha_apertu = df_prestamos_tra[['prestamo_id','sucursal_id','fecha_otorgamiento']].copy()
df_veri_fecha_sucur = df_sucursales_tra.copy()
df_merge_veri_fecha_apertu = df_veri_fecha_apertu.merge(
    right= df_veri_fecha_sucur,
    on="sucursal_id",
    how='left'
)
# Verifica que la fecha_otorgamiento no sea mayor a la fecha_apertura 
# Resultados Esperados: Tabla Vacia 
df_merge_veri_fecha_apertu[
    (df_merge_veri_fecha_apertu.fecha_otorgamiento 
        < df_merge_veri_fecha_apertu.fecha_apertura)
]

,prestamo_id,sucursal_id,fecha_otorgamiento,fecha_apertura


### fecha_ingreso (relacion con oficial de credito)

In [96]:
# LEFT JOIN, por oficial_id 
df_veri_fecha_ingreso = df_prestamos_tra[['prestamo_id','oficial_id','fecha_otorgamiento']].copy()
df_veri_fecha_ingreso_ofi = df_oficial_tra.copy()
df_merge_veri_fecha_ingreso = df_veri_fecha_ingreso.merge(
    right= df_veri_fecha_ingreso_ofi,
    on="oficial_id",
    how='left'
)
# Verifica que la fecha_otorgamiento no sea mayor a la fecha_apertura 
# Resultados Esperados: Tabla Vacia 
df_revisar_fecha_ingreso = df_merge_veri_fecha_ingreso[
                                        (df_merge_veri_fecha_ingreso.fecha_otorgamiento 
                                            < df_merge_veri_fecha_ingreso.fecha_ingreso)
                                        ].copy()
df_revisar_fecha_ingreso

,prestamo_id,oficial_id,fecha_otorgamiento,fecha_ingreso
5,6,5,2020-08-07,2022-08-04
8,9,26,2020-02-20,2021-11-29
13,14,50,2022-01-02,2022-04-16
29,30,52,2022-12-30,2024-03-13
32,33,45,2020-06-05,2023-05-20
...,...,...,...,...
6469,6470,54,2022-05-17,2024-02-16
6482,6483,52,2022-02-08,2024-03-13
6487,6488,28,2020-11-26,2023-02-03
6488,6489,10,2020-09-14,2024-10-01


In [97]:
df_revisar_fecha_ingreso["ERROR"] = "FECHA DE OTORGAMIENTO DEL CREDITO ATERIOR A LA FECHA DE INGRESO DEL OFICIAL"

# Limpiando Duplicados Luego de Limpieza

In [98]:
# Muestra registros duplicados
df_prestamos_tra[df_prestamos_tra.duplicated(keep=False)]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real
201,202,2531,17,5,52,CONT-00000202,2023-08-28,2025-05-19,6363.54,1954.81,...,5,Vigente,0,Normal,Aval,0.00,Electrodomésticos,Transferencia Bancaria,2023-09-27,2024-12-20
871,872,23,6,5,40,CONT-00000872,2021-11-20,2024-02-08,4271.14,1234.00,...,0,Cancelado,354,Pérdida,Aval,0.00,Muebles y Enseres,Cheque de Gerencia,2021-12-20,2024-02-08
1679,1680,1419,6,3,41,CONT-00001680,2020-11-10,2026-04-13,33391.44,11075.01,...,16,Vigente,0,Normal,n/a,62577.71,Maquinaria,Agencia,2020-12-10,2024-12-19
4164,4165,2821,23,1,60,CONT-00004165,2021-02-13,2024-07-27,6207.18,0.00,...,0,Cancelado,0,Normal,Sin Garantía,0.00,n/a,Transferencia Bancaria,2021-03-15,2024-07-27
6495,4165,2821,23,1,60,CONT-00004165,2021-02-13,2024-07-27,6207.18,0.00,...,0,Cancelado,0,Normal,Sin Garantía,0.00,n/a,Transferencia Bancaria,2021-03-15,2024-07-27
6496,872,23,6,5,40,CONT-00000872,2021-11-20,2024-02-08,4271.14,1234.00,...,0,Cancelado,354,Pérdida,Aval,0.00,Muebles y Enseres,Cheque de Gerencia,2021-12-20,2024-02-08
6497,202,2531,17,5,52,CONT-00000202,2023-08-28,2025-05-19,6363.54,1954.81,...,5,Vigente,0,Normal,Aval,0.00,Electrodomésticos,Transferencia Bancaria,2023-09-27,2024-12-20
6498,1680,1419,6,3,41,CONT-00001680,2020-11-10,2026-04-13,33391.44,11075.01,...,16,Vigente,0,Normal,n/a,62577.71,Maquinaria,Agencia,2020-12-10,2024-12-19


In [99]:
# Elimina duplicados y guarda el ultimo registro porque, se consedera que es el mas actualizado 
df_prestamos_tra.drop_duplicates(keep='last',inplace=True)

## prestamo_id

In [100]:
df_prestamos_tra[df_prestamos_tra.prestamo_id.duplicated()]

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real


# Exportando la Tabla Limpia

In [101]:
df_prestamos_tra.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6495 entries, 0 to 6498
Data columns (total 29 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   prestamo_id                   6495 non-null   int64         
 1   cliente_id                    6495 non-null   int64         
 2   sucursal_id                   6495 non-null   int64         
 3   producto_id                   6495 non-null   int64         
 4   oficial_id                    6495 non-null   int64         
 5   numero_contrato               6495 non-null   object        
 6   fecha_otorgamiento            6495 non-null   datetime64[ns]
 7   fecha_vencimiento             6495 non-null   datetime64[ns]
 8   monto_original                6495 non-null   float64       
 9   saldo_capital_vigente         6495 non-null   float64       
 10  tasa_interes_nominal_anual    6495 non-null   float64       
 11  tasa_interes_efectiva_anual   6495 

In [102]:
df_prestamos_tra.head()

,prestamo_id,cliente_id,sucursal_id,producto_id,oficial_id,numero_contrato,fecha_otorgamiento,fecha_vencimiento,monto_original,saldo_capital_vigente,...,numero_cuotas_pendientes,estado,dias_mora,clasificacion_riesgo_sbs,garantia_tipo,garantia_valor,proposito_credito,canal_desembolso,fecha_primer_pago_programado,fecha_ultimo_pago_real
0,1,4848,23,6,60,CONT-00000001,2023-05-07,2025-10-23,9218.73,3596.11,...,10,Vigente,0,Normal,n/a,0.0,Viaje / Turismo,Transferencia Bancaria,2023-06-06,2024-12-27
1,2,44,5,1,21,CONT-00000002,2021-02-12,2025-07-21,7767.38,1473.65,...,7,Vigente,0,Normal,Aval,0.0,Mejoras del Hogar,Agencia,2021-03-14,2024-12-23
2,3,2474,16,1,35,CONT-00000003,2023-11-22,2027-11-01,12512.04,10640.44,...,35,Vigente,0,Normal,Aval,0.0,Mejoras del Hogar,Transferencia Bancaria,2023-12-22,2024-12-16
3,4,637,15,4,32,CONT-00000004,2024-04-19,2024-10-16,3555.11,0.00,...,0,Cancelado,0,Normal,Carta Fianza,0.0,Expansión de Negocio,Agencia,2024-05-19,2024-10-16
4,5,3622,10,4,12,CONT-00000005,2020-04-19,2022-10-06,8388.15,0.00,...,0,Cancelado,0,Normal,Sin Garantía,0.0,Capital de Trabajo,Agencia,2020-05-19,2022-10-06


In [103]:
df_prestamos_tra.to_parquet(
    obtener_ruta_archivo("archivos_semi_limpios","semi_limpio_prestamos.parquet"),
    index = False
)

# Exportando los Registros de Prestamos a Revisar

In [104]:
# Guardando estos registros en el data set para verrificar 
df_para_verificar = pd.concat([df_revisar_garantia,df_revisar_fecha_otor,df_revisar_fecha_ingreso])
df_para_verificar.to_parquet(
    obtener_ruta_archivo("archivos_para_revision","revision_prestamos.parquet"),
    index=False
)